In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-21")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
race_results_list = spark.read.parquet(f"{presentation_path}/race_results")

In [0]:
display(race_results_list)

In [0]:
race_results_list = spark.read.parquet(f"{presentation_path}/race_results") \
    .filter(f"race_date = '{v_file_date}'") \
    .select("race_year") \
    .distinct() \
    .collect()

In [0]:
race_year_list = []
for race_year in race_results_list:
    race_year_list.append(race_year.race_year)


In [0]:
driver_standings_df = race_results_df \
    .groupBy("race_year","driver_name","driver_nationality", "team") \
    .agg(
        sum("points").alias("total_points"),
        count(when(col("position") == 1, True)).alias("wins")
    )


In [0]:
display(driver_standings_df)

In [0]:
driver_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"), desc("wins"))

In [0]:
final_df = driver_standings_df.withColumn("rank", dense_rank().over(driver_rank_spec))


In [0]:
display(final_df.filter("race_year = 2020"))

In [0]:
final_df.write.mode("overwrite").parquet(f"{presentation_path}/driver_standings")